# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Notebook repaired and passed a complete diagnostic T4 run (15/15 cells, 0 errors, AP 85.8 test bbox, counts produced), but the 3-session budget was exhausted before a fresh-runtime Run-all validation could be repeated; diagnostic success does not replace fresh validation, so no publication. Requeue path: allocate one fresh T4 session and exec deliverables/p-26ff2584da9abdb7eb0c9266d19fc880.ipynb (sha256 9778620364fbb1f9d1bd9276a3d799789d88e1d405b56c6ca36395cc075aa206), archive *_output.ipynb + CLI log, then publish. Fixes applied this invocation: pip arg order, zip layout (no wrapping folder), non-interactive split acceptance with printed proposal, idempotent COCOEvaluator patch, output pre-cleanup.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: GinJinn2: Object detection and segmentation for ecology and evolution

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-26ff2584da9abdb7eb0c9266d19fc880)


# GinJinn2 — Seed-mixture detection and per-genus counting (minimal Colab reproduction)

**Paper:** Ott, T. & Lautenschlager, U. (2022). *GinJinn2: Object detection and segmentation for ecology and evolution.* Methods in Ecology and Evolution **13**(3), 603–610. [DOI 10.1111/2041-210X.13787](https://doi.org/10.1111/2041-210X.13787)

**Tool code:** [AGOberprieler/GinJinn2](https://github.com/AGOberprieler/GinJinn2) @ commit `96e1cefa86aad66b82f7e02e7444912077ee72a5` (Apache-2.0) · **Config:** `seeds.yaml`, Zenodo record [5747582](https://doi.org/10.5281/zenodo.5747582) · **Data:** GFBio/BGBM dataset [0035](https://data.bgbm.org/dataset/gfbio/0035/) "GinJinn2 - seeds dataset" (CC0, 285 microscopic seed-mixture images with COCO bounding-box annotations, genera *Arabidopsis* and *Sedum*).

**Scope (partial demonstration):** This notebook re-runs the paper's seed-mixture example end to end — data preparation, train/val/test split, training the deposited `seeds.yaml` Faster R-CNN configuration, test-set evaluation, prediction with visualization, and per-genus counting. **The model is trained for only 300 iterations** (the deposited configuration specifies 7000; we shorten it with the authors' own `-n/--n_iter` CLI override) so the whole pipeline fits in one Colab session. The result is a **pipeline demonstration with an under-trained model**: detection quality will be far below the published results, and no paper-trained weights exist to compare against (the authors did not archive any).

**実行環境:必ず `ランタイム` → `ランタイムのタイプを変更` で **T4 GPU** を選択してから実行してください。** GinJinn2/Detectron2 の学習は GPU 前提であり、CPU では実行できません(著者ドキュメントも NVidia GPU を必須要件として明記)。想定所要時間:約 25〜40 分(ダウンロード 約 755 MiB を含む)。

**Expected output (概要):** 学習済み(300 iteration)モデルによるテスト画像の種子検出、COCO AP 指標、属ごとの種子数カウント表。

## 1. Runtime check — verify a GPU is allocated

We first verify that a CUDA GPU is visible and print the exact hardware. GinJinn2 is a GPU tool (its authors list an NVidia GPU as a hard requirement, and the training path of Detectron2 is impractical on CPU), so this notebook **fails loudly instead of silently falling back to CPU**.

**GPU が割り当てられているかを確認します。** 見つからない場合はエラーで停止します(CPU への黙ってのフォールバックは行いません)。

In [ ]:
import subprocess, sys

nvidia_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(nvidia_smi.stdout[-1200:] if nvidia_smi.returncode == 0 else nvidia_smi.stderr)
if nvidia_smi.returncode != 0:
    raise RuntimeError(
        "No GPU visible. GinJinn2 requires an NVIDIA GPU.\n"
        "In Colab: Runtime > Change runtime type > Hardware accelerator = T4 GPU.\n"
        "(GPU が見つかりません。ランタイムタイプを T4 GPU に変更してください。)"
    )

import torch, platform
print("Python:", platform.python_version())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "torch cannot see the GPU" 

## 2. Install dependencies

The notebook installs everything it needs into the fresh Colab VM (nothing is pre-baked):
- `ninja`, `opencv`, `pycocotools`, `imantics`, `pandas` — ginjinn's runtime dependencies for the train/evaluate/predict/count path (enumerated from the pinned source: `ginjinn/trainer/trainer.py`, `ginjinn/predictor/predictors.py`, `ginjinn/data_reader/*.py` import `pandas`, `matplotlib`, `cv2`, `pycocotools`, `imantics`).
- `setuptools` guard: ginjinn's `ginjinn new` command uses `pkg_resources`, which very new setuptools releases removed.

We also define a small `run()` helper used by every later cell: it prints the command, captures its output (so the full log is visible in the notebook), and raises immediately on a non-zero exit.

**依存パッケージをインストールします。** `pkg_resources` が使えるよう setuptools のバージョンも調整します。

In [ ]:
import subprocess, sys

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, **kw)
    if r.stdout: print(r.stdout[-4000:])
    if r.stderr: print("[stderr]", r.stderr[-2000:])
    if r.returncode != 0:
        raise RuntimeError(f"command failed (rc={r.returncode}): {' '.join(str(c) for c in cmd)}")
    return r

def pip(*args):
    return run([sys.executable, "-m", "pip", *args])

try:
    import pkg_resources  # noqa: F401
    print("pkg_resources available")
except ImportError:
    pip("install", "-q", "setuptools<81")
    import pkg_resources  # noqa: F401
    print("pkg_resources installed via setuptools<81")

# cv2 and pycocotools are preinstalled on current Colab images; verify before use.
import cv2, pycocotools  # noqa: F401
print("cv2", cv2.__version__)
pip("install", "-q", "imantics", "pandas", "matplotlib")
import imantics  # noqa: F401
print("imantics OK") 

## 3. Install Detectron2 (built from source)

GinJinn2 is a command-line wrapper around [Detectron2](https://github.com/facebookresearch/detectron2). Detectron2 has no wheel for current PyTorch/CUDA combinations, so the standard Colab route is a source build (`--no-build-isolation` is required because Detectron2 needs torch at build time). **This compiles CUDA kernels and takes roughly 10–20 minutes**; the build summary is printed below.

**Detectron2 をソースからビルドします(約 10〜20 分)。** GinJinn2 は Detectron2 の CLI ラッパーです。

In [ ]:
pip("install", "--no-build-isolation", "ninja",
    "git+https://github.com/facebookresearch/detectron2.git") 

## 4. Install GinJinn2 at the pinned author revision

We install ginjinn directly from the author repository at the exact commit inspected for this reproduction (`96e1cefa86aa`, Apache-2.0). GinJinn2's `setup.py` declares no dependencies (they were installed above) and registers the `ginjinn` console command. We verify the installation with the tool's own help page.

**著者リポジトリの固定コミットから GinJinn2 をインストールし、`ginjinn -h` で動作を確認します。

In [ ]:
pip("install", "--no-build-isolation", "--quiet",
    "git+https://github.com/AGOberprieler/GinJinn2.git@96e1cefa86aad66b82f7e02e7444912077ee72a5")

r = run(["ginjinn", "-h"])
assert "train" in r.stdout, "ginjinn CLI not installed" 

## 5. Documented compatibility patch for current Detectron2

GinJinn2 was written against Detectron2 v0.4 (2021). In Detectron2 v0.6+ the `COCOEvaluator` constructor changed from positional `(dataset_name, cfg, distributed, output_dir)` to keyword-only arguments, so ginjinn's `ValTrainer.build_evaluator` (`ginjinn/trainer/trainer.py`, line 151 in the pinned revision) must be adapted. This is the **only executable-code change** made to the author tool; everything else runs unmodified. The patch below rewrites the single call in the installed package, shows the diff, and fails if the pattern is not found exactly once (it is safe to re-run: an already-patched installation is detected).

**互換性パッチ:** 現行 Detectron2 では `COCOEvaluator` の引数形式が変わったため、インストール済み `trainer.py` の 1 行をキーワード引数形式に書き換えます(本 reproduction で行う唯一のコード変更)。再実行しても安全です。

In [ ]:
import ginjinn, pathlib

trainer_path = pathlib.Path(ginjinn.__file__).parent / "trainer" / "trainer.py"
src = trainer_path.read_text()
old = "return COCOEvaluator(dataset_name, cfg, True, output_folder)"
new = "return COCOEvaluator(dataset_name, output_dir=output_folder)"
if src.count(old) == 1:
    trainer_path.write_text(src.replace(old, new))
    print("Patched:", trainer_path)
elif new in src:
    print("Already patched:", trainer_path)
else:
    raise RuntimeError("COCOEvaluator call site not found in installed ginjinn trainer.py")
print("--- applied form ---")
print("    " + new) 

## 6. Download the GFBio seeds dataset (≈755 MiB, CC0)

The dataset `gfbio0035.zip` (285 microscopic seed-mixture images + COCO `annotations.json`, genera *Arabidopsis* / *Sedum*, license CC0) is downloaded **inside this Colab VM** from the Botanic Garden and Botanical Museum Berlin data archive, then validated: expected file count (285 images per the archive's dataset page) and annotation categories are checked from the actual bytes before any model work. The verified archive layout is `annotations.json` + `images/` (+ `eml.xml`, `LICENSES_AND_CITATION.txt`), without a wrapping folder.

**データセットを Colab 内にダウンロードし、検証します**(285 枚の画像と COCO アノテーションを期待通り確認)。

In [ ]:
from pathlib import Path
import zipfile, json, shutil

base = Path("/content/ginjinn"); base.mkdir(parents=True, exist_ok=True)
zip_path = base / "gfbio0035.zip"
ds_url = "https://data.bgbm.org/dataset/gfbio/0035/gfbio0035.zip"
if not zip_path.exists():
    print("downloading", ds_url, flush=True)
    run(["wget", "-q", "-O", str(zip_path), ds_url])

print("zip size (MiB):", round(zip_path.stat().st_size / 2**20, 1))
extract_dir = base / "gfbio0035"
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path) as zf:  # safe member check before extraction
        bad = [n for n in zf.namelist() if n.startswith("/") or ".." in Path(n).parts]
        assert not bad, f"unsafe zip members: {bad[:5]}"
        # verified layout: annotations.json, images/ (285 jpg), eml.xml, LICENSES_AND_CITATION.txt
        zf.extractall(extract_dir)

ann_path = extract_dir / "annotations.json"
assert ann_path.is_file(), "annotations.json missing from extracted dataset"
img_dir = extract_dir / "images"
images = sorted(p.name for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print("dataset root:", extract_dir)
print("images:", len(images))
assert len(images) == 285, f"expected 285 images, found {len(images)}"

coco = json.loads(ann_path.read_text())
cats = {c["id"]: c["name"] for c in coco["categories"]}
print("categories:", cats)
print("annotations:", len(coco["annotations"]), "| images in annotations:", len(coco["images"])) 

## 7. Input preview — one seed-mixture image with its real COCO annotations

Before any model work we render one training image (the one with the most annotated seeds) together with its **reference bounding-box annotations** from `annotations.json`. Boxes are colored by genus; these are the authors' ground-truth labels from the GFBio archive, not model predictions.

**入力データの確認:** 参照(GT)アノテーションを重ねた実画像を表示します。これはモデル予測ではなく、GFBio アーカイブの正解ラベルです。

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

coco_api = COCO(str(ann_path))
img_id = max(coco_api.imgToAnns, key=lambda i: len(coco_api.imgToAnns[i]))  # image with most seeds
img_info = coco_api.loadImgs(img_id)[0]
img = cv2.cvtColor(cv2.imread(str(img_dir / img_info["file_name"])), cv2.COLOR_BGR2RGB)

cat_colors = {cid: plt.cm.tab10(i % 10)[:3] for i, cid in enumerate(sorted(cats))}
for ann in coco_api.imgToAnns[img_id]:
    x, y, w, h = ann["bbox"]
    c = (np.array(cat_colors[ann["category_id"]]) * 255).astype(int)
    cv2.rectangle(img, (int(x), int(y)), (int(x + w), int(y + h)), tuple(int(v) for v in c), 3)

plt.figure(figsize=(9, 7))
plt.imshow(img); plt.axis("off")
legend = [plt.Line2D([0], [0], color=np.array(cat_colors[cid]) / 255, lw=4,
                     label=f"{name} (reference annotation)") for cid, name in sorted(cats.items())]
plt.legend(handles=legend, loc="lower right", framealpha=0.9)
plt.title(f'GFBio 0035 sample: {img_info["file_name"]} — reference COCO bounding boxes')
plt.tight_layout(); plt.savefig("/content/gt_preview.png", dpi=110, bbox_inches="tight")
plt.show()
print(f"image {img_info['file_name']}: {len(coco_api.imgToAnns[img_id])} reference boxes") 

## 8. Data preparation — flatten and train/val/test split

Following the paper's documented example workflow (repo docs, *Example Applications → Seed Counting*): `ginjinn utils flatten` normalizes the COCO dataset to the flat layout ginjinn expects, and `ginjinn split` partitions it 60% train / 20% val / 20% test with the category-balanced heuristic from `ginjinn/data_reader/data_splitter.py`.

**Non-interactive note:** the pinned `ginjinn split` prints its category-balance *split proposal* and asks the user to accept it (or reject and retry). In this non-interactive notebook we accept the first proposal programmatically (`"y"` on stdin), **print the proposal table below for review**, and verify the resulting per-split counts — the same decision a user makes at the terminal, recorded in the output.

**論文の Example workflow どおりに flatten と 60/20/20 分割を実行します。** `ginjinn split` は対話的に分割案の承認を求めるため、ノートブックでは stdin 経由で承認し、承認した分割表を出力に表示して後から確認できるようにしています。

In [ ]:
flat = base / "seeds_flat"
split = base / "seeds_split"
for d in (flat, split):  # fresh split on every run; the CLI would otherwise prompt
    if d.exists():
        shutil.rmtree(d)

run(["ginjinn", "utils", "flatten", "-i", str(img_dir), "-a", str(ann_path), "-o", str(flat)])
# 'y' answers the split-proposal confirmation (retried if no valid split is found);
# the printed proposal and per-split counts below document what was accepted.
run(["ginjinn", "split", "-I", str(flat), "-o", str(split),
     "-d", "bbox-detection", "-t", "0.2", "-v", "0.2"], input="y\n" * 10)

for sub in ["train", "val", "test"]:
    n_img = len(list((split / sub / "images").iterdir()))
    n_ann = len(json.loads((split / sub / "annotations.json").read_text())["annotations"])
    print(f"{sub}: {n_img} images, {n_ann} annotations") 

## 9. Project configuration — the authors' deposited `seeds.yaml`

We fetch the deposited configuration file `seeds.yaml` (Zenodo [5747582](https://doi.org/10.5281/zenodo.5747582), the authors' "commands and configuration of exemplary analyses") inside Colab, **verify its MD5 against the value recorded during source inspection** (`bd4d21941cefc833f6f4514c861681d8`), and install it as the project's `ginjinn_config.yaml`. Only two mechanical changes are made: `project_dir` is set to this Colab location, and the relative input paths (`../seeds_split/...`) resolve because we create the project directory as a sibling of `seeds_split`, exactly like the deposited layout. The model is `faster_rcnn_R_101_FPN_3x` with `weights: pretrained`, i.e. training starts from the Detectron2 model-zoo ImageNet/COCO-pretrained backbone (fetched automatically), since the authors archived no paper-trained weights.

**Zenodo から `seeds.yaml` を取得し MD5 を検証した上で、プロジェクト設定として書き出します。** 変更は `project_dir` のみ(機械的なパス修正)です。

In [ ]:
seeds_yaml_url = "https://zenodo.org/api/records/5747582/files/seeds.yaml/content"
expected_md5 = "bd4d21941cefc833f6f4514c861681d8"
cfg_src = base / "seeds.yaml"
run(["wget", "-q", "-O", str(cfg_src), seeds_yaml_url])
import hashlib
digest = hashlib.md5(cfg_src.read_bytes()).hexdigest()
print("seeds.yaml md5:", digest)
assert digest == expected_md5, f"seeds.yaml md5 mismatch: {digest}"

project = base / "seeds_split_analysis"
(project / "outputs").mkdir(parents=True, exist_ok=True)
cfg_text = cfg_src.read_text().replace("project_dir: /home/ginjinn/seeds_split_analysis",
                                       f"project_dir: {project}")
assert "project_dir: /home" not in cfg_text, "project_dir rewrite failed"
(project / "ginjinn_config.yaml").write_text(cfg_text)
print("wrote", project / "ginjinn_config.yaml")
print(cfg_text[:400], "\n...") 

## 10. Train — the deposited configuration, shortened to 300 iterations

This runs the authors' training command with the deposited configuration. Because the full 7000 iterations would take hours, we use ginjinn's own documented override `ginjinn train -n/--n_iter` (`argument_parser/train.py`) to stop after **300 iterations**. Training starts from the Detectron2 model-zoo `faster_rcnn_R_101_FPN_3x` weights (auto-downloaded by `get_checkpoint_url`, ≈250 MiB), uses the deposited augmentations (flips, brightness/contrast/saturation, ±30° rotation) and evaluates on the validation set every 250 iterations. Any previous outputs are removed first so the demo is reproducible and the deposited `resume: True` option never resumes stale state. **Expect ~3–6 minutes on T4.**

**学習を実行します。** 論文の設定から、著者提供の `-n` オーバーライドで 300 iteration に短縮しています(フル学習は数千 iteration・数時間規模)。モデルは Detectron2 model zoo の事前学習重みから開始します。

In [ ]:
outputs_dir = project / "outputs"
if outputs_dir.exists():
    shutil.rmtree(outputs_dir)  # fresh training run; deposited resume option would otherwise resume
outputs_dir.mkdir(parents=True, exist_ok=True)

print("ginjinn train -n 300 (deposited seeds.yaml, shortened run)", flush=True)
run(["ginjinn", "train", str(project), "-n", "300"]) 

## 11. Training curves from `outputs/metrics.json`

GinJinn writes its Detectron2 training log to `outputs/metrics.json` (the paper's workflow monitors the rendered `metrics.pdf`). We plot the total training loss and the validation bounding-box AP recorded during the shortened run, so the under-training is visible rather than hidden.

**学習ログから loss と検証 AP の曲線を描画します。** 300 iteration の短縮実行であるため、loss はまだ下がり途中です。

In [ ]:
import pandas as pd, json

metrics_file = project / "outputs" / "metrics.json"
rows = [json.loads(l) for l in metrics_file.read_text().splitlines() if l.strip()]
metrics = pd.DataFrame(rows).sort_values("iteration")
print("recorded iterations:", metrics["iteration"].min(), "-", metrics["iteration"].max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
loss_cols = [c for c in metrics.columns if c.endswith("loss") or "total_loss" in c]
for c in loss_cols:
    axes[0].plot(metrics["iteration"], metrics[c], label=c, alpha=0.8)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("loss"); axes[0].legend(fontsize=7)
axes[0].set_title("Training losses (300-iter demo run)")
ap_cols = [c for c in metrics.columns if c.startswith("bbox/")]
for c in ap_cols:
    axes[1].plot(metrics["iteration"], metrics[c], marker="o", label=c)
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("AP (val)")
axes[1].set_title("Validation bbox AP (val split)")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig("/content/training_curves.png", dpi=110); plt.show() 

## 12. Evaluate on the test split

`ginjinn evaluate` loads `outputs/model_final.pth` and computes COCO detection metrics on the held-out test split (`ginjinn/evaluation/evaluate.py` uses `COCOEvaluator` + `inference_on_dataset`). The printed `bbox/AP` values are **those of the 300-iteration demonstration model** — they are expected to be clearly worse than any converged model and are not comparable to published paper results (none are published for a comparable 300-iter run).

**テスト分割に対する COCO AP を計算します。** 値は 300 iteration のデモモデルのものです。

In [ ]:
run(["ginjinn", "evaluate", str(project)])

eval_csv = project / "evaluation.csv"
if eval_csv.exists():
    df = pd.read_csv(eval_csv)
    display(df.T.rename(columns={0: "value"}))
else:
    print("evaluation.csv not found in", project) 

## 13. Predict on a few test images with visualization

We run `ginjinn predict` with the trained demo model on three held-out test images and turn on `-v/--visualize`, which renders predicted bounding boxes onto the images (saved under the prediction's `visualization/` folder). The confidence threshold is lowered to 0.5 (`-t/--threshold`, author CLI flag) because the 300-iteration model is under-trained. **These are model predictions, clearly distinct from the reference annotations shown in section 7.**

**テスト画像 3 枚に対して予測を実行し、可視化します。** これはモデル予測であり、第 7 節の参照アノテーションとは区別されます。

In [ ]:
pred_out = base / "seeds_test_prediction"
if pred_out.exists():
    shutil.rmtree(pred_out)  # avoid the CLI's interactive overwrite prompt
test_imgs = sorted((split / "test" / "images").iterdir())[:3]
pred_in = base / "pred_sample"
if pred_in.exists():
    shutil.rmtree(pred_in)
pred_in.mkdir()
for p in test_imgs:
    shutil.copy(p, pred_in / p.name)
print("predicting on:", [p.name for p in test_imgs])

run(["ginjinn", "predict", str(project), "-i", str(pred_in),
     "-o", str(pred_out), "-v", "-t", "0.5"])

vis_files = sorted((pred_out / "visualization").iterdir())
print("visualizations:", [p.name for p in vis_files])
pred_json = json.loads((pred_out / "annotations.json").read_text())
print("predicted boxes:", len(pred_json["annotations"])) 

## 14. Show prediction overlays

The rendered predictions from `ginjinn predict -v` are displayed side by side. Boxes and labels are drawn by ginjinn's own visualizer (`ginjinn/predictor/predictors.py`, using Detectron2's `Visualizer`); the caption notes that these come from the 300-iteration demo model.

**予測結果の可視化を表示します。** 描画は ginjinn 組み込みの Visualizer によるものです。

In [ ]:
n = len(vis_files)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
if n == 1: axes = [axes]
for ax, p in zip(axes, vis_files):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(f"prediction: {p.name}\n(300-iter demo model, threshold 0.5)", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.savefig("/content/prediction_overlays.png", dpi=110); plt.show() 

## 15. Per-genus seed counting — the paper's target output

The scientific goal of this example is quantifying seed-mixture composition: `ginjinn utils count` reads the COCO predictions and counts objects per category (genus) per image, writing a CSV (`commandline_app/utils/count.py` → `count_categories`). We display the counts for the three predicted images and export `seed_counts.csv` for the learner.

**論文の目的である属ごとの種子数カウントを実行します。** 結果は CSV にも出力されます。

In [ ]:
counts_csv = base / "seed_counts.csv"
if counts_csv.exists():
    counts_csv.unlink()  # avoid the CLI's interactive overwrite prompt
run(["ginjinn", "utils", "count", "-a", str(pred_out / "annotations.json"),
     "-o", str(counts_csv), "-s"])

counts = pd.read_csv(counts_csv).set_index("image")
display(counts)
print("\nper-genus totals (demo model, 3 test images):")
print(counts.sum()) 

## 16. Interpretation, limitations, and references

**What was demonstrated.** The complete GinJinn2 seed-mixture workflow of the paper — data preparation (flatten + split), project creation from the authors' deposited `seeds.yaml`, Faster R-CNN training from Detectron2 model-zoo weights, test-set COCO evaluation, prediction with visualization, and per-genus counting — executed end to end on one Colab T4 session with the authors' CLI.

**What was NOT demonstrated.**
- Converged performance: 300 iterations instead of the deposited 7000 (hours of GPU time); expect clearly worse AP and noisy counts than a real analysis.
- No paper-trained weights exist (the authors archived none), so there is no reference checkpoint to match against — results are retrained, not checkpoint-identical (this is inherent to the paper's own release).
- Only 3 of the test images are predicted/counted to keep the session short.
- Two documented non-interactive adaptations (both recorded in the notebook text): the split-proposal confirmation is answered programmatically with the proposal printed for review, and a one-line compatibility patch to ginjinn's `ValTrainer.build_evaluator` (section 5) was required for current Detectron2; the paper's own evaluation path was already compatible.

**制限事項:** 本 notebook は 300 iteration の短縮デモであり、論文の収束結果を再現するものではありません。著者は学習済み重みを公開していないため、比較対象のチェックポイントも存在しません。

**References**
- Ott & Lautenschlager (2022), MEE 13(3):603–610, DOI 10.1111/2041-210X.13787.
- GinJinn2 code: https://github.com/AGOberprieler/GinJinn2 @ `96e1cefa86aad66b82f7e02e7444912077ee72a5` (Apache-2.0).
- Configuration: Zenodo 5747582 `seeds.yaml` (MD5 bd4d21941cefc833f6f4514c861681d8).
- Data: GFBio/BGBM dataset 0035 (CC0), https://data.bgbm.org/dataset/gfbio/0035/ .
- Wu, Y. et al. Detectron2, https://github.com/facebookresearch/detectron2 .